# 🫀 Phân tích Dữ liệu Bệnh Tim

Notebook này phân tích dataset dự đoán bệnh tim và huấn luyện mô hình Machine Learning.

In [ ]:
# Import thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Load và Khám phá Dữ liệu

In [ ]:
# Load dataset
df = pd.read_csv('../dataset/heart_disease_processed.csv')
print(f"📊 Dataset shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")
df.head()

In [ ]:
# Thông tin dataset
print("📈 Thông tin Dataset:")
print("="*50)
df.info()

In [ ]:
# Thống kê mô tả
df.describe()

In [ ]:
# Kiểm tra missing values
print("🔍 Missing Values:")
print(df.isnull().sum())

## 2. Phân tích Trực quan (EDA)

In [ ]:
# Phân bố Target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
target_counts = df['target'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(target_counts, labels=['Không bệnh', 'Có bệnh'], autopct='%1.1f%%', 
            colors=colors, explode=[0.02, 0.02])
axes[0].set_title('Phân bố Target', fontweight='bold')

# Bar chart
sns.countplot(data=df, x='target', palette=colors, ax=axes[1])
axes[1].set_xlabel('Target (0: Không bệnh, 1: Có bệnh)')
axes[1].set_ylabel('Số lượng')
axes[1].set_title('Số lượng theo Target', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Phân bố tuổi theo Target
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x='age', hue='target', kde=True, palette=['#2ecc71', '#e74c3c'])
plt.title('Phân bố Tuổi theo Kết quả Bệnh tim', fontweight='bold')
plt.xlabel('Tuổi')
plt.ylabel('Số lượng')
plt.legend(['Không bệnh', 'Có bệnh'])
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f')
plt.title('Ma trận Tương quan giữa các Features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tương quan với Target
target_corr = correlation['target'].drop('target').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in target_corr.values]
target_corr.plot(kind='barh', color=colors)
plt.title('Tương quan với Target', fontweight='bold')
plt.xlabel('Hệ số tương quan')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Huấn luyện Mô hình

In [ ]:
# Chuẩn bị dữ liệu
X = df.drop('target', axis=1)
y = df['target']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 Training set: {len(X_train)} samples")
print(f"📊 Test set: {len(X_test)} samples")

In [ ]:
# Huấn luyện Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

print("🔄 Đang huấn luyện mô hình...")
model.fit(X_train, y_train)
print("✅ Hoàn tất!")

In [ ]:
# Đánh giá mô hình
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"🎯 Accuracy: {accuracy:.2%}")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Không bệnh', 'Có bệnh']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Không bệnh', 'Có bệnh'],
            yticklabels=['Không bệnh', 'Có bệnh'])
plt.title('Confusion Matrix', fontweight='bold')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5)
print(f"📊 Cross-validation scores: {cv_scores}")
print(f"📊 Mean CV score: {cv_scores.mean():.2%} (+/- {cv_scores.std()*2:.2%})")

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='#3498db')
plt.xlabel('Importance')
plt.title('Feature Importance - Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Lưu Mô hình

In [ ]:
# Lưu model
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/heart_model.pkl')
print("💾 Đã lưu mô hình: models/heart_model.pkl")

## 5. Thử nghiệm Dự đoán

In [ ]:
# Thử dự đoán với mẫu mới
sample = {
    'age': 55,
    'sex': 1,
    'cp': 2,
    'trestbps': 140,
    'chol': 250,
    'fbs': 0,
    'restecg': 1,
    'thalach': 150,
    'exang': 0,
    'oldpeak': 1.5,
    'slope': 1,
    'ca': 0,
    'thal': 2
}

sample_df = pd.DataFrame([sample])
prediction = model.predict(sample_df)[0]
probability = model.predict_proba(sample_df)[0]

print("🧪 Kết quả dự đoán mẫu thử:")
print(f"   Dự đoán: {'Có nguy cơ bệnh tim' if prediction == 1 else 'Không có nguy cơ'}")
print(f"   Xác suất không bệnh: {probability[0]:.2%}")
print(f"   Xác suất có bệnh: {probability[1]:.2%}")

---
**© 2026 Daivid AI - Hệ thống Dự đoán Bệnh Tim**